# NHANES project about Periodontal disease and Geriatric Nutrition Risk Index (GNRI): descriptive and regression analysis
> This notebook has the purpose to collect all the analysis on Nhanes dataset for a medical paper project 

Requirements and Information:
1. Nhanes dataset from 2009/10 to 2013/14
2. Outcome:
    - Geriatric Nutrition Risk Index (GNRI)
3. Exposure:
    - number of teeth (OHXDEN)
    - consists of 2 categories: patients with < 20 teeth, patients with >= 20 teeth
    - Other categorization:
        1. Edentulus : 0 teeth
        2. Severe Loss : 1-9 teeth
        3. Moderate Loss : 10-19 teeth
        4. Nearly Complete : >=20 teeth
4. Confounding Variables:
    - Gender (RIAGENDR)
    - Age at screening (RIDAGEYR)
    - Race (RIDRETH1)
    - Education	(DMDEDUC2)
    - Poverty income ratio (INDFMPIR)
    - Smoking status (SMQ020)
    - Alchool intake (ALQ101)
5. Mediators:
    - Heart failure	(RIDRETH1)  
    - Coronary heart disease (MCQ160b)
    - Stroke (MCQ160c)
    - Liver disease	(MCQ160o)
    - Cancer (MCQ220)
    - Diabetes (DIQ010)
    - High blood pressure (BPQ020)
6. Age => 60

## Import Libraries

In [1]:
library(haven)
library(nhanesA)
library(survey)
library(MASS)
library(dplyr)
library(tidyr)
library(tidyverse)
library(ggplot2)
library(readr)
library(flextable)
library(officer)
library(nnet)
library(broom)
library(ggplot2)

Loading required package: grid

Loading required package: Matrix

Loading required package: survival


Attaching package: 'survey'


The following object is masked from 'package:graphics':

    dotchart



Attaching package: 'dplyr'


The following object is masked from 'package:MASS':

    select


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'tidyr'


The following objects are masked from 'package:Matrix':

    expand, pack, unpack


-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v forcats   1.0.0     v readr     2.1.5
v ggplot2   3.5.1     v stringr   1.5.1
v lubridate 1.9.4     v tibble    3.2.1
v purrr     1.0.2     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x tidyr::expand() masks Matrix::expand()
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stat

## Configurations

In [2]:
path_to_data_09_10 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2009_10/"
path_to_data_11_12 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2011_12/"
path_to_data_13_14 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2013_14/"

## Load Dataset & Feature Selection

In [3]:
# Datasets for 2009/10 period

demo_09_10 <- read_xpt(file.path(path_to_data_09_10, "DEMO_F.xpt"))

demo_09_10_selected <- demo_09_10 %>%
  select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)

alcohol_09_10 <- read_xpt(file.path(path_to_data_09_10, "ALQ_F.xpt"))

alcohol_09_10_selected <- alcohol_09_10 %>%
  select(SEQN, ALQ101)

smoking_09_10 <- read_xpt(file.path(path_to_data_09_10, "SMQ_F.xpt.txt"))

smoking_09_10_selected <- smoking_09_10 %>%
    select(SEQN, SMQ020)

med_conditions_09_10 <- read_xpt(file.path(path_to_data_09_10, "MCQ_F.xpt"))

med_conditions_09_10_selected <- med_conditions_09_10 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)


blood_pressure_09_10 <- read_xpt(file.path(path_to_data_09_10, "BPQ_F.xpt"))

blood_pressure_09_10_selected <- blood_pressure_09_10 %>%
    select(SEQN, BPQ020)


diabetes_09_10 <- read_xpt(file.path(path_to_data_09_10, "DIQ_F.xpt"))

diabetes_09_10_selected <- diabetes_09_10 %>%
    select(SEQN, DIQ010)


teeth_09_10 <- read_xpt(file.path(path_to_data_09_10, "OHXDEN_F.xpt.txt"))

selected_cols <- colnames(teeth_09_10)[grepl("^OHX\\d{2}TC", colnames(teeth_09_10))]

teeth_09_10_selected <- teeth_09_10 %>%
    select(SEQN, all_of(selected_cols))


albumin_09_10 <- read_xpt(file.path(path_to_data_09_10, "BIOPRO_F.xpt.txt"))

albumin_09_10_selected <- albumin_09_10 %>%
    select(SEQN, LBDSALSI)

w_h_09_10 <- read_xpt(file.path(path_to_data_09_10, "BMX_F.xpt"))

w_h_09_10_selected <- w_h_09_10 %>%
    select(SEQN, BMXWT, BMXHT)

In [4]:
# Datasets for 2011/12 period

demo_11_12 <- read_xpt(file.path(path_to_data_11_12, "DEMO_G.xpt.txt"))

demo_11_12_selected <- demo_11_12 %>%
  select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)

alcohol_11_12 <- read_xpt(file.path(path_to_data_11_12, "ALQ_G.xpt.txt"))

alcohol_11_12_selected <- alcohol_11_12 %>%
  select(SEQN, ALQ101)


smoking_11_12 <- read_xpt(file.path(path_to_data_11_12, "SMQ_G.xpt.txt"))

smoking_11_12_selected <- smoking_11_12 %>%
    select(SEQN, SMQ020)


med_conditions_11_12 <- read_xpt(file.path(path_to_data_11_12, "MCQ_G.xpt.txt"))

med_conditions_11_12_selected <- med_conditions_11_12 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)


blood_pressure_11_12 <- read_xpt(file.path(path_to_data_11_12, "BPQ_G.xpt.txt"))

blood_pressure_11_12_selected <- blood_pressure_11_12 %>%
    select(SEQN, BPQ020)


diabetes_11_12 <- read_xpt(file.path(path_to_data_11_12, "DIQ_G.xpt.txt"))

diabetes_11_12_selected <- diabetes_11_12 %>%
    select(SEQN, DIQ010)


teeth_11_12 <- read_xpt(file.path(path_to_data_11_12, "OHXDEN_G.xpt.txt"))

selected_cols <- colnames(teeth_11_12)[grepl("^OHX\\d{2}TC", colnames(teeth_11_12))]

teeth_11_12_selected <- teeth_11_12 %>%
    select(SEQN, all_of(selected_cols))


albumin_11_12 <- read_xpt(file.path(path_to_data_11_12, "BIOPRO_G.xpt.txt"))

albumin_11_12_selected <- albumin_11_12 %>%
    select(SEQN, LBDSALSI)

w_h_11_12 <- read_xpt(file.path(path_to_data_11_12, "BMX_G.xpt.txt"))

w_h_11_12_selected <- w_h_11_12 %>%
    select(SEQN, BMXWT, BMXHT)

In [5]:
# Datasets for 2013/14 period

demo_13_14 <- read_xpt(file.path(path_to_data_13_14, "DEMO_H.xpt.txt"))

demo_13_14_selected <- demo_13_14 %>%
    select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)


alcohol_13_14 <- read_xpt(file.path(path_to_data_13_14, "ALQ_H.xpt.txt"))

alcohol_13_14_selected <- alcohol_13_14 %>%
    select(SEQN, ALQ101)


smoking_13_14 <- read_xpt(file.path(path_to_data_13_14, "SMQ_H.xpt.txt"))

smoking_13_14_selected <- smoking_13_14 %>%
    select(SEQN, SMQ020)


med_conditions_13_14 <- read_xpt(file.path(path_to_data_13_14, "MCQ_H.xpt.txt"))

med_conditions_13_14_selected <- med_conditions_13_14 %>%
    select(SEQN, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)


blood_pressure_13_14 <- read_xpt(file.path(path_to_data_13_14, "BPQ_H.xpt.txt"))

blood_pressure_13_14_selected <- blood_pressure_13_14 %>%
    select(SEQN, BPQ020)


diabetes_13_14 <- read_xpt(file.path(path_to_data_13_14, "DIQ_H.xpt.txt"))

diabetes_13_14_selected <- diabetes_13_14 %>%
    select(SEQN, DIQ010)


teeth_13_14 <- read_xpt(file.path(path_to_data_13_14, "OHXDEN_H.xpt.txt"))

selected_cols <- colnames(teeth_13_14)[grepl("^OHX\\d{2}TC", colnames(teeth_13_14))]

teeth_13_14_selected <- teeth_13_14 %>%
    select(SEQN, all_of(selected_cols))


albumin_13_14 <- read_xpt(file.path(path_to_data_13_14, "BIOPRO_H.xpt.txt"))

albumin_13_14_selected <- albumin_13_14 %>%
    select(SEQN, LBDSALSI)

w_h_13_14 <- read_xpt(file.path(path_to_data_13_14, "BMX_H.xpt.txt"))

w_h_13_14_selected <- w_h_13_14 %>%
    select(SEQN, BMXWT, BMXHT)

## Merge datasets without NA and missing values
> Merge all data from each datasets and then exclude patients

In [6]:
# Merge datasets demographics and intrinsic capacity data

datasets_09_10 <- list(
  demo_09_10_selected, alcohol_09_10_selected, smoking_09_10_selected, med_conditions_09_10_selected,
  blood_pressure_09_10_selected, diabetes_09_10_selected,
  albumin_09_10_selected, w_h_09_10_selected, teeth_09_10_selected
)

datasets_11_12 <- list(
  demo_11_12_selected, alcohol_11_12_selected, smoking_11_12_selected, med_conditions_11_12_selected,
  blood_pressure_11_12_selected, diabetes_11_12_selected,
  albumin_11_12_selected, w_h_11_12_selected, teeth_11_12_selected
)

datasets_13_14 <- list(
  demo_13_14_selected, alcohol_13_14_selected, smoking_13_14_selected, med_conditions_13_14_selected,
  blood_pressure_13_14_selected, diabetes_13_14_selected,
  albumin_13_14_selected, w_h_13_14_selected, teeth_13_14_selected
)

# Horizontal union for period 2009/10, 2011/12, 2013/14

df_09_10 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_09_10)

df_11_12 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_11_12)

df_13_14 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_13_14)

# Vertical union

df_final <- bind_rows(df_09_10, df_11_12, df_13_14)

print("Dimensions before removing NA values")
dim(df_final)

# Filter with AGE >= 60

df_final_age_60 <- subset(df_final, RIDAGEYR >= 60)

print("Dimensions with AGE >= 60")
dim(df_final_age_60)

# Excluding patients with missing values in features required for GNRI calculation (weight, height, albumin)

df_final_excluding_GNRI <- df_final_age_60[complete.cases(df_final_age_60[, c('BMXWT', 'BMXHT', 'LBDSALSI')]), ]

print("Dimensions without GNRI missing values")
dim(df_final_excluding_GNRI)

# Excluding patients with no examinations for Teeth counts

df_final_excluding_teeth <- df_final_excluding_GNRI %>%
  filter(rowSums(!is.na(select(., starts_with("OHX")))) > 0)

print("Dimensions without Teeth counts missing values")
dim(df_final_excluding_teeth)

# Excluding patients with missing values in Confounding features

df_final_excluding_confounding <- df_final_excluding_teeth[complete.cases(df_final_excluding_teeth[, 
                                  c('RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'ALQ101', 'SMQ020', 'MCQ160B',
                                  'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ160L', 'MCQ220', 'BPQ020', 'DIQ010')]), ]

df_final_merged <- df_final_excluding_confounding %>%
  filter(!if_any(c(DMDEDUC2, ALQ101, SMQ020, MCQ160B, MCQ160C, MCQ160D,
                   MCQ160E, MCQ160F, MCQ160L, MCQ220, BPQ020, DIQ010), ~ . == 9))

df_final_merged <- df_final_merged %>%
  filter(!if_any(c(DMDEDUC2, ALQ101, SMQ020, MCQ160B, MCQ160C, MCQ160D,
                   MCQ160E, MCQ160F, MCQ160L, MCQ220, BPQ020, DIQ010), ~ . == 7))

print("Dimensions without Confounding missing values")
dim(df_final_merged)

[1] "Dimensions before removing NA values"


[1] 30468    52

[1] "Dimensions with AGE >= 60"


[1] 5705   52

[1] "Dimensions without GNRI missing values"


[1] 5008   52

[1] "Dimensions without Teeth counts missing values"


[1] 4372   52

[1] "Dimensions without Confounding missing values"


[1] 3725   52

In [7]:
# for cycle to see table for each categorical variable
categorical_vars <- c('RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'ALQ101', 'SMQ020', 
                      'MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 
                      'MCQ160F', 'MCQ160L', 'MCQ220',
                      'BPQ020', 'DIQ010')
for (var in categorical_vars) {
  print(paste("Table for", var))
  print(table(df_final_merged[[var]], useNA = "ifany"))
}

[1] "Table for RIAGENDR"

   1    2 
1852 1873 
[1] "Table for RIDRETH1"

   1    2    3    4    5 
 370  341 1943  790  281 
[1] "Table for DMDEDUC2"

   1    2    3    4    5 
 491  536  857 1020  821 
[1] "Table for ALQ101"

   1    2 
2503 1222 
[1] "Table for SMQ020"

   1    2 
1901 1824 
[1] "Table for MCQ160B"

   1    2 
 236 3489 
[1] "Table for MCQ160C"

   1    2 
 332 3393 
[1] "Table for MCQ160D"

   1    2 
 175 3550 
[1] "Table for MCQ160E"

   1    2 
 313 3412 
[1] "Table for MCQ160F"

   1    2 
 281 3444 
[1] "Table for MCQ160L"

   1    2 
 183 3542 
[1] "Table for MCQ220"

   1    2 
 766 2959 
[1] "Table for BPQ020"

   1    2 
2323 1402 
[1] "Table for DIQ010"

   1    2    3 
 860 2708  157 


In [9]:
teeth_cols <- grep("^OHX\\d{2}TC$", names(df_final_merged), value = TRUE)

for (var in teeth_cols) {
  print(paste("Table for", var))
  print(table(df_final_merged[[var]], useNA = "ifany"))
}

[1] "Table for OHX01TC"

   2    4    5 
 448 3263   14 
[1] "Table for OHX02TC"

   2    3    4    5 
1736    5 1942   42 
[1] "Table for OHX03TC"

   2    3    4    5 
1763   20 1886   56 
[1] "Table for OHX04TC"

   2    3    4    5 
1922   15 1732   56 
[1] "Table for OHX05TC"

   2    3    4    5 
1972   13 1672   68 
[1] "Table for OHX06TC"

   1    2    3    4    5 
   2 2450    9 1204   60 
[1] "Table for OHX07TC"

   1    2    3    4    5 
   1 2264   12 1390   58 
[1] "Table for OHX08TC"

   2    3    4    5 
2276    5 1397   47 
[1] "Table for OHX09TC"

   2    3    4    5 
2296   11 1372   46 
[1] "Table for OHX10TC"

   1    2    3    4    5 
   1 2290    5 1368   61 
[1] "Table for OHX11TC"

   2    3    4    5 
2468    9 1197   51 
[1] "Table for OHX12TC"

   2    3    4    5 
1979   16 1642   88 
[1] "Table for OHX13TC"

   2    3    4    5 
1909   18 1734   64 
[1] "Table for OHX14TC"

   2    3    4    5 
1781   19 1884   41 
[1] "Table for OHX15TC"

   2    3    4   

In [13]:
# check how many patients have one or more teeth_cols with value 1

find_patients_with_1 <- function(df) {
  teeth_cols <- grep("^OHX\\d{2}TC$", names(df), value = TRUE)
  
  no_teeth_patients <- df[rowSums(df[, teeth_cols] == 1, na.rm = TRUE) > 0, ]
  
  return(no_teeth_patients)
}

prova <- find_patients_with_1(df_final_merged)
dim(prova)

[1]  8 52

In [14]:
# Saving completed and cleaned dataframe for analysis

write.csv(df_final_merged, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/merged_and_cleaned_df_09_14_GNRI_teeth.csv", row.names = FALSE)

## Teeth counts

Preprocessed features:
- total number of teeth
- binary category: >=20 teeth or < 20 teeth
- edentulus category
- Other categorization:
    1. Edentulus : 0 teeth
    2. Severe Loss : 1-9 teeth
    3. Moderate Loss : 10-19 teeth
    4. Nearly Complete : >=20 teeth

In [40]:
# Functions to check if there are patients with zero permanent teeth,
# and moreover, patients with only not present teeth and fragments/root

find_patients_no_teeth <- function(df) {

  teeth_cols <- grep("^OHX\\d{2}TC$", names(df), value = TRUE)
  
  no_teeth_patients <- df[rowSums(df[, teeth_cols] == 2, na.rm = TRUE) == 0, ]
  
  return(no_teeth_patients)
}

find_patients_with_non4_values <- function(df) {
  
  teeth_cols <- grep("^OHX\\d{2}TC$", names(df), value = TRUE)

  no_teeth_patients <- find_patients_no_teeth(df)

  print("Dimensions of patients with no teeth")
  print(dim(no_teeth_patients))

  patients_with_non4 <- no_teeth_patients[rowSums(no_teeth_patients[, teeth_cols] == 4, na.rm = TRUE) == 0, ]

  if (dim(patients_with_non4)[1] != 0) {
     print("Dimensions of patients with non-4 values in teeth columns")
     print(dim(patients_with_non4))
  } else {
     print("Dimensions of patients with non-4 values in teeth columns are empty")
  }

  return(patients_with_non4)

}

test_edentolus_no_4 <- find_patients_with_non4_values(df_final_merged)

test_edentolus <- find_patients_no_teeth(df_final_merged)

# funzione per sapere per ogni paziente quanti denti hanno valori differenti da 4 (no table)

i = 1
j <- 0
for (patient in test_edentolus$SEQN) {
   teeth_cols <- grep("^OHX\\d{2}TC$", names(test_edentolus), value = TRUE)
   
   non4_count <- rowSums(test_edentolus[i, teeth_cols] != 4, na.rm = TRUE)

   if (non4_count > 0) {
      print(paste("Patient SEQN:", patient, "has", non4_count, "teeth with non-4 values."))
      j <- j + 1
   }
   i <- i + 1
}
print(paste("Total patients with non-4 teeth values:", j))

[1] "Dimensions of patients with no teeth"
[1] 631  52
[1] "Dimensions of patients with non-4 values in teeth columns are empty"


[1] "Patient SEQN: 52967 has 4 teeth with non-4 values."
[1] "Patient SEQN: 54070 has 5 teeth with non-4 values."
[1] "Patient SEQN: 57992 has 1 teeth with non-4 values."
[1] "Patient SEQN: 58127 has 6 teeth with non-4 values."
[1] "Patient SEQN: 59615 has 6 teeth with non-4 values."
[1] "Patient SEQN: 60998 has 3 teeth with non-4 values."
[1] "Patient SEQN: 63134 has 2 teeth with non-4 values."
[1] "Patient SEQN: 64552 has 6 teeth with non-4 values."
[1] "Patient SEQN: 64973 has 2 teeth with non-4 values."
[1] "Patient SEQN: 65637 has 5 teeth with non-4 values."
[1] "Patient SEQN: 65713 has 2 teeth with non-4 values."
[1] "Patient SEQN: 67022 has 3 teeth with non-4 values."
[1] "Patient SEQN: 68377 has 5 teeth with non-4 values."
[1] "Patient SEQN: 68966 has 7 teeth with non-4 values."
[1] "Patient SEQN: 69001 has 9 teeth with non-4 values."
[1] "Patient SEQN: 69505 has 5 teeth with non-4 values."
[1] "Patient SEQN: 71020 has 2 teeth with non-4 values."
[1] "Patient SEQN: 71739 has 2 

In [48]:
# Function to calculate total number of teeth for each patient and categorize it

count_teeth <- function(df) {

  teeth_cols <- grep("^OHX\\d{2}TC$", names(df), value = TRUE)
  
  df$total_teeth <- rowSums(df[, teeth_cols] == 2, na.rm = TRUE)
  
  # Binary category: 1 if >=20 teeth, 0 otherwise
  df$has_20_or_more_teeth <- ifelse(df$total_teeth >= 20, 1, 0)
  
  # edentulous patients (every 32 teeth with value 4 or 5 or 3)

  df$edentulous <- ifelse(rowSums(df[, teeth_cols] == 4 | df[, teeth_cols] == 5 | df[, teeth_cols] == 3, na.rm = TRUE) == length(teeth_cols), 1, 0)
  
  # Other possible categories: Edentulous, Severe, Moderate, Nearly Complete
  df$teeth_category <- cut(
    df$total_teeth,
    breaks = c(-Inf, 0, 9, 19, 32),
    labels = c("Edentulous", "Severe Loss (1-9)", "Moderate Loss (10-19)", "Nearly Complete (20-32)"),
    right = TRUE
  )
  
  df <- df[, !names(df) %in% teeth_cols]
  
  return(df)
}

df <- as.data.frame(count_teeth(df_final_merged))
head(df)
dim(df)
table(df$teeth_category, useNA = "ifany")

,SEQN,RIAGENDR,RIDAGEYR,RIDRETH1,DMDEDUC2,INDFMPIR,ALQ101,SMQ020,MCQ160B,MCQ160C,...,MCQ220,BPQ020,DIQ010,LBDSALSI,BMXWT,BMXHT,total_teeth,has_20_or_more_teeth,edentulous,teeth_category
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,51628,2,60,4,3,0.69,2,1,2,2,...,2,1,1,39,116.8,166.0,25,1,0,Nearly Complete (20-32)
2,51633,1,80,3,4,1.27,1,1,2,2,...,2,2,2,43,79.1,174.3,6,0,0,Severe Loss (1-9)
3,51635,1,80,3,2,1.69,2,1,2,1,...,2,2,1,43,89.6,180.1,0,0,1,Edentulous
4,51645,1,66,1,2,0.41,1,2,2,2,...,2,1,2,44,82.9,171.3,26,1,0,Nearly Complete (20-32)
5,51654,1,66,3,4,2.20,1,1,2,2,...,2,1,2,39,68.0,169.5,26,1,0,Nearly Complete (20-32)
6,51661,2,60,1,3,2.75,1,2,2,2,...,2,2,2,40,73.5,151.4,23,1,0,Nearly Complete (20-32)


[1] 3725   24


             Edentulous       Severe Loss (1-9)   Moderate Loss (10-19) 
                    631                     406                     686 
Nearly Complete (20-32) 
                   2002 

In [49]:
# Saving preprocessed

write.csv(df, "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/preprocessed_df_GNRI_teeth_09_14.csv", row.names=FALSE)

## Descriptive Analysis